# Operation Advisor: Learning & Uncertainty Benchmark
**NUS ISS Practice Module Group 31**  
This notebook evaluates supervised fault classifiers (KNN baseline, Decision Tree, Random Forest, MLP), unsupervised operating regime clustering, and Bayesian Belief Network inference.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in python path
sys.path.append(os.path.abspath('..'))

from src.learning.classifiers import ScenarioClassifierSuite, FEATURE_COLUMNS
from src.learning.bayesian_network import ProcessBayesianNetwork
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

## 1. Dataset Loading & Feature Exploration

In [ ]:
data_path = os.path.join('..', 'data', 'simulation', 'telemetry_timeseries.csv')
df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape}")
display(df.head())
print("\nClass Distribution:")
print(df['scenario_label'].value_counts())

## 2. Supervised Scenario Classifiers Benchmark
Evaluating **KNN (Baseline)**, **Decision Tree (Interpretable Rules)**, **Random Forest (Ensemble)**, and **MLP (Neural Network)**.

In [ ]:
suite = ScenarioClassifierSuite(random_state=42)
evaluations = suite.train(df)

metrics_df = pd.DataFrame([
    {
        'Model': name,
        'Macro F1': ev.macro_f1,
        'Precision': ev.precision,
        'Recall': ev.recall
    }
    for name, ev in evaluations.items()
])
display(metrics_df)

## 3. Feature Importance Analysis (Random Forest)

In [ ]:
importances = suite.get_feature_importances()
imp_df = pd.DataFrame(list(importances.items()), columns=['Feature', 'Gini Importance'])
display(imp_df)

## 4. Unsupervised Clustering for Operating Regime Discovery

In [ ]:
X = df[FEATURE_COLUMNS].fillna(0.0).values
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X)

print(f"Explained Variance by 2 Principal Components: {pca.explained_variance_ratio_.sum():.2%}")

## 5. Bayesian Belief Network: Probabilistic Diagnosis & Entropy Reduction

In [ ]:
bn = ProcessBayesianNetwork()

# Case: High Pressure and Low Gas Export Flow
evidence = {'SYM_HIGH_PRESSURE': 1, 'SYM_LOW_GAS_FLOW': 1}
result = bn.infer(evidence)

print(f"Diagnostic Shannon Entropy: {result.shannon_entropy:.4f} bits (Normalized: {result.normalized_entropy:.2%})")
print(f"Recommended Next Check for Maximal Info Gain: {result.most_informative_next_check}")

ranking_df = pd.DataFrame([
    {
        'Hypothesis': h.hypothesis_id,
        'Description': h.description,
        'Posterior Probability': f"{h.posterior_prob:.2%}",
        'Prior Probability': f"{h.prior_prob:.2%}",
        'Likelihood Ratio': h.likelihood_ratio
    }
    for h in result.ranked_hypotheses
])
display(ranking_df)